In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

TEST_DIR = ROOT / "outputs" / "test"

OUT_DIR = TEST_DIR / "baseline_vs_candidates.csv"

align_data = pd.read_csv(
    TEST_DIR / "overall_alignscore_results.csv"
)

summac_rouge_data = pd.read_csv(
    TEST_DIR / "summac_rouge_eval.csv"
)

In [ ]:
# Merge candidate scores with the selected candidate information
selected = align_selected[
    ["id", "selected_candidate", "selected_summary"]
].copy()

selected = selected.merge(
    align_candidates,
    on="id",
    how="inner"
)

selected = selected.merge(
    summac_candidates,
    on="id",
    how="inner"
)

# Extract scores corresponding to the selected candidate
for metric in ["alignscore", "summac", "rougeL"]:
    selected[f"selected_{metric}"] = selected.apply(
        lambda row: row[
            f"{metric}_{int(row['selected_candidate'])}"
        ],
        axis=1
    )

In [ ]:

import pandas as pd
from pathlib import Path

# ==========================================
# 1. PATHS
# ==========================================

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

TEST_DIR = ROOT / "outputs" / "test"

OUT_PATH = TEST_DIR / "baseline_vs_candidates.csv"


# ==========================================
# 2. LOAD CSV FILES
# ==========================================

align_data = pd.read_csv(
    TEST_DIR / "overall_alignscore_results.csv"
)

summac_rouge_data = pd.read_csv(
    TEST_DIR / "summac_rouge_eval.csv"
)


# ==========================================
# 3. NORMALIZE IDS
# ==========================================

align_data["id"] = align_data["id"].astype(str)
summac_rouge_data["id"] = summac_rouge_data["id"].astype(str)


# ==========================================
# 4. CHECK FOR DUPLICATE IDS
# ==========================================

if align_data["id"].duplicated().any():
    raise ValueError("Duplicate IDs in AlignScore CSV.")

if summac_rouge_data["id"].duplicated().any():
    raise ValueError("Duplicate IDs in SummaC/ROUGE CSV.")


# ==========================================
# 5. SELECT REQUIRED COLUMNS
# ==========================================

align_columns = align_data[
    [
        "id",
        "baseline_summary",
        "selected_summary",
        "baseline_alignscore",
        "selected_alignscore",
        "selected_candidate",
        "alignscore_1",
        "alignscore_2",
        "alignscore_3",
        "alignscore_4",
        "alignscore_5"
    ]
].copy()


summac_columns = summac_rouge_data[
    [
        "id",
        "baseline_summary",
        "candidate_summary",
        "baseline_summac",
        "baseline_rougeL",
        "candidate_summac",
        "candidate_rougeL"
    ]
].copy()


# ==========================================
# 6. MERGE CSV FILES
# ==========================================

# Keep the AlignScore summaries as the main summaries.
# SummaC/ROUGE supplies the corresponding evaluation scores.

final_df = align_columns.merge(
    summac_columns[
        [
            "id",
            "baseline_summac",
            "baseline_rougeL",
            "candidate_summac",
            "candidate_rougeL"
        ]
    ],
    on="id",
    how="inner",
    validate="one_to_one"
)


# ==========================================
# 7. CHECK MERGED DATA
# ==========================================

if len(final_df) != len(align_data):
    raise ValueError(
        "Some AlignScore article IDs were not matched with SummaC/ROUGE."
    )

if final_df.isnull().any().any():
    print("Warning: Missing values detected in the merged data.")


# ==========================================
# 8. CALCULATE SCORE DIFFERENCES
# ==========================================

# Difference = selected candidate - baseline

final_df["alignscore_difference"] = (
    final_df["selected_alignscore"] -
    final_df["baseline_alignscore"]
)

final_df["summac_difference"] = (
    final_df["candidate_summac"] -
    final_df["baseline_summac"]
)

final_df["rougeL_difference"] = (
    final_df["candidate_rougeL"] -
    final_df["baseline_rougeL"]
)


# ==========================================
# 9. RENAME SUMMARY COLUMNS
# ==========================================

final_df = final_df.rename(
    columns={
        "selected_summary": "candidate_summary"
    }
)


# ==========================================
# 10. ARRANGE FINAL COLUMNS
# ==========================================

final_df = final_df[
    [
        "id",
        "baseline_summary",
        "selected_candidate_summary",
        "selected_candidate",
        "baseline_alignscore",
        "selected_alignscore",
        "baseline_summac",
        "candidate_summac",
        "baseline_rougeL",
        "candidate_rougeL",
        "alignscore_difference",
        "summac_difference",
        "rougeL_difference",

    ]
]


# ==========================================
# 11. SAVE CSV
# ==========================================

final_df.to_csv(
    OUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Comparison CSV saved:", OUT_PATH)
print("Total articles:", len(final_df))
print("Total columns:", len(final_df.columns))


# ==========================================
# 12. DISPLAY AVERAGE SCORES
# ==========================================

print("\nAVERAGE BASELINE SCORES")

print(
    final_df[
        [
            "baseline_alignscore",
            "baseline_summac",
            "baseline_rougeL"
        ]
    ].mean()
)

print("\nAVERAGE CANDIDATE SCORES")

print(
    final_df[
        [
            "selected_alignscore",
            "candidate_summac",
            "candidate_rougeL"
        ]
    ].mean()
)

print("\nAVERAGE SCORE DIFFERENCES")

print(
    final_df[
        [
            "alignscore_difference",
            "summac_difference",
            "rougeL_difference"
        ]
    ].mean()
)

display(final_df.head())


Comparison CSV saved: c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\outputs\test\baseline_vs_candidates.csv
Total articles: 615
Total columns: 13

AVERAGE BASELINE SCORES
baseline_alignscore    0.918327
baseline_summac        0.761174
baseline_rougeL        0.367518
dtype: float64

AVERAGE CANDIDATE SCORES
selected_alignscore    0.991045
candidate_summac       0.875307
candidate_rougeL       0.365842
dtype: float64

AVERAGE SCORE DIFFERENCES
alignscore_difference    0.072717
summac_difference        0.114132
rougeL_difference       -0.001676
dtype: float64


,id,baseline_summary,candidate_summary,selected_candidate,baseline_alignscore,selected_alignscore,baseline_summac,candidate_summac,baseline_rougeL,candidate_rougeL,alignscore_difference,summac_difference,rougeL_difference
0,SJ1Xmf-Rb,"We propose FearNet, a generative model that do...",We propose FearNet for incremental class learn...,2,0.995088,0.998429,0.728941,0.992083,0.195122,0.250000,0.003341,0.263142,0.054878
1,HJDUjKeA-,We show how discrete objects can be learnt in ...,We show how discrete objects can be learnt in ...,1,0.991177,0.991177,0.993535,0.993535,1.000000,1.000000,0.000000,0.000000,0.000000
2,BJgLg3R9KQ,We extend a state-of-the-art attention network...,We extend a state-of-the-art attention network...,2,0.990473,0.994491,0.108901,0.091573,0.245614,0.245614,0.004018,-0.017328,0.000000
3,BklpOo09tQ,We show that adversarial examples can be gener...,We show that adversarial examples can be gener...,2,0.980715,0.980883,0.995010,0.994702,0.228571,0.202899,0.000167,-0.000309,-0.025673
4,BJLmN8xRW,We compare the effectiveness of various deep l...,We perform an empirical comparison for the imp...,1,0.983012,0.996886,0.104666,0.994226,0.228571,0.263158,0.013874,0.889560,0.034586


In [ ]:

import pandas as pd
from pathlib import Path

# ==========================================
# 1. PATHS
# ==========================================

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

TEST_DIR = ROOT / "outputs" / "test"

OUT_PATH = TEST_DIR / "baseline_vs_candidates.csv"


# ==========================================
# 2. LOAD CSV FILES
# ==========================================

align_data = pd.read_csv(
    TEST_DIR / "overall_alignscore_results.csv"
)

summac_rouge_data = pd.read_csv(
    TEST_DIR / "summac_rouge_eval.csv"
)


# ==========================================
# 3. NORMALIZE IDS
# ==========================================

align_data["id"] = align_data["id"].astype(str)
summac_rouge_data["id"] = summac_rouge_data["id"].astype(str)


# ==========================================
# 4. CHECK FOR DUPLICATE IDS
# ==========================================

if align_data["id"].duplicated().any():
    raise ValueError("Duplicate IDs in AlignScore CSV.")

if summac_rouge_data["id"].duplicated().any():
    raise ValueError("Duplicate IDs in SummaC/ROUGE CSV.")


# ==========================================
# 5. SELECT REQUIRED COLUMNS
# ==========================================

align_columns = align_data[
    [
        "id",
        "baseline_summary",
        "selected_summary",
        "baseline_alignscore",
        "selected_alignscore",
        "selected_candidate",
        "alignscore_1",
        "alignscore_2",
        "alignscore_3",
        "alignscore_4",
        "alignscore_5"
    ]
].copy()


summac_columns = summac_rouge_data[
    [
        "id",
        "baseline_summary",
        "candidate_summary",
        "baseline_summac",
        "baseline_rougeL",
        "candidate_summac",
        "candidate_rougeL"
    ]
].copy()


# ==========================================
# 6. MERGE CSV FILES
# ==========================================

# Keep the AlignScore summaries as the main summaries.
# SummaC/ROUGE supplies the corresponding evaluation scores.

final_df = align_columns.merge(
    summac_columns[
        [
            "id",
            "baseline_summac",
            "baseline_rougeL",
            "candidate_summac",
            "candidate_rougeL"
        ]
    ],
    on="id",
    how="inner",
    validate="one_to_one"
)


# ==========================================
# 7. CHECK MERGED DATA
# ==========================================

if len(final_df) != len(align_data):
    raise ValueError(
        "Some AlignScore article IDs were not matched with SummaC/ROUGE."
    )

if final_df.isnull().any().any():
    print("Warning: Missing values detected in the merged data.")


# ==========================================
# 8. CALCULATE SCORE DIFFERENCES
# ==========================================

# Difference = selected candidate - baseline

final_df["alignscore_difference"] = (
    final_df["selected_alignscore"] -
    final_df["baseline_alignscore"]
)

final_df["summac_difference"] = (
    final_df["candidate_summac"] -
    final_df["baseline_summac"]
)

final_df["rougeL_difference"] = (
    final_df["candidate_rougeL"] -
    final_df["baseline_rougeL"]
)


# ==========================================
# 9. RENAME SUMMARY COLUMNS
# ==========================================

final_df = final_df.rename(
    columns={
        "selected_summary": "candidate_summary"
    }
)


# ==========================================
# 10. ARRANGE FINAL COLUMNS
# ==========================================

final_df = final_df[
    [
        "id",

        # Summaries
        "baseline_summary",
        "candidate_summary",
        "selected_candidate",

        # AlignScore
        "baseline_alignscore",
        "selected_alignscore",
        "alignscore_difference",

        # SummaC
        "baseline_summac",
        "candidate_summac",
        "summac_difference",

        # ROUGE-L
        "baseline_rougeL",
        "candidate_rougeL",
        "rougeL_difference",

        # Individual candidate AlignScores
        "alignscore_1",
        "alignscore_2",
        "alignscore_3",
        "alignscore_4",
        "alignscore_5"
    ]
]


# ==========================================
# 11. SAVE CSV
# ==========================================

final_df.to_csv(
    OUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Comparison CSV saved:", OUT_PATH)
print("Total articles:", len(final_df))
print("Total columns:", len(final_df.columns))


# ==========================================
# 12. DISPLAY AVERAGE SCORES
# ==========================================

print("\nAVERAGE BASELINE SCORES")

print(
    final_df[
        [
            "baseline_alignscore",
            "baseline_summac",
            "baseline_rougeL"
        ]
    ].mean()
)

print("\nAVERAGE CANDIDATE SCORES")

print(
    final_df[
        [
            "selected_alignscore",
            "candidate_summac",
            "candidate_rougeL"
        ]
    ].mean()
)

print("\nAVERAGE SCORE DIFFERENCES")

print(
    final_df[
        [
            "alignscore_delta",
            "summac_delta",
            "rougeL_delta"
        ]
    ].mean()
)

display(final_df.head())


Comparison CSV saved: c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\outputs\test\baseline_vs_candidates.csv
Total articles: 615
Total columns: 18

AVERAGE BASELINE SCORES
baseline_alignscore    0.918327
baseline_summac        0.761174
baseline_rougeL        0.367518
dtype: float64

AVERAGE CANDIDATE SCORES
selected_alignscore    0.991045
candidate_summac       0.875307
candidate_rougeL       0.365842
dtype: float64

AVERAGE SCORE DIFFERENCES


KeyError: "None of [Index(['alignscore_delta', 'summac_delta', 'rougeL_delta'], dtype='str')] are in the [columns]"